In [12]:
import pandas as pd

# ======================
# LOAD DATA
# ======================
small = pd.read_excel(r"E:\bankruptcy-risk-system\data\raw\small_cap_india_screener.xlsx")
mid = pd.read_excel(r"E:\bankruptcy-risk-system\data\raw\mid_cap_india_screener.xlsx")
large = pd.read_excel(r"E:\bankruptcy-risk-system\data\raw\large_cap_india_screener.xlsx")

world = pd.read_csv(r"E:\bankruptcy-risk-system\data\raw\world_bank.csv")
industry = pd.read_excel(r"E:\bankruptcy-risk-system\data\raw\industry_data.xlsx")
management = pd.read_excel(r"E:\bankruptcy-risk-system\data\raw\management_data.xlsx")

# ======================
# MERGE FINANCIAL DATA
# ======================
financial = pd.concat([small, mid, large], ignore_index=True)
financial.columns = financial.columns.str.strip()

financial.rename(columns={
    "Name": "company_name",
    "Debt / Eq": "debt_to_equity",
    "Current ratio": "current_ratio",
    "ROA 12M %": "roa",
    "CF Operations Rs.Cr.": "cash_flow",
    "Debt Rs.Cr.": "debt"
}, inplace=True)

# ======================
# BUSINESS RISK (WORLD)
# ======================
world = world[["country", "year", "inflation"]]
world = world[world["country"] == "India"]
world_latest = world.sort_values("year").iloc[-1]

financial["inflation"] = world_latest["inflation"]

# ======================
# INDUSTRY DATA
# ======================
industry_avg = industry.mean(numeric_only=True)

for col in industry_avg.index:
    financial[col] = industry_avg[col]

# ======================
# MANAGEMENT DATA (SAFE HANDLING)
# ======================
management.columns = management.columns.str.strip()

# Auto-detect company column
possible_cols = ["Company Name", "Name", "Borrower Name"]

company_col = None
for col in possible_cols:
    if col in management.columns:
        company_col = col
        break

if company_col:
    management.rename(columns={company_col: "company_name"}, inplace=True)
    management["default_flag"] = 1

    merged = financial.merge(
        management[["company_name", "default_flag"]],
        on="company_name",
        how="left"
    )

    merged["default_flag"] = merged["default_flag"].fillna(0)

else:
    print("⚠️ Company column not found in management data → skipping merge")
    merged = financial.copy()
    merged["default_flag"] = 0

# ======================
# SAVE
# ======================
import os

BASE_DIR = r"E:\bankruptcy-risk-system"
os.makedirs(os.path.join(BASE_DIR, "data/processed"), exist_ok=True)

merged.to_csv(os.path.join(BASE_DIR, "data/processed/merged_data.csv"), index=False)